In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import (
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier as KNN

from sklearn.metrics import mean_absolute_error, mean_squared_error
import os
import sys
sys.path.append(os.path.abspath(".."))
import src.feature_engineering as fe
from sklearn import datasets
from sklearn.model_selection import train_test_split


import mlflow
from mlflow.models import infer_signature
import mlflow.pyfunc

## MLFlow on my dataset

In [3]:
df = pd.read_parquet(
    '../data/final_datasets/datasets_linear_models/conso_v3_linear.parquet'
)

In [4]:
last_year = df['year'].iloc[-1]
last_year

np.int32(2024)

In [255]:
df.columns

Index(['Consommation', 'Zone_A', 'Zone_B', 'Zone_C',
       'Vacances de la Toussaint', 'Vacances de Noël', 'Vacances d'Hiver',
       'Vacances de Printemps', 'Vacances d'Été', 'public_holidays', '44T',
       '69T', '59T', '75T', '13T', '33T', 'T', 'U', 'FF', 'PMER', 'RR1',
       'year', 'month', 'hour', 'day_of_week', 'is_weekend', 'hour_sin',
       'hour_cos', 'day_of_week_sin', 'day_of_week_cos', 'month_sin',
       'month_cos', 'lagged_1', 'lagged_2', 'lagged_48', 'lagged_336',
       'rolling_mean_24h', 'rolling_std_24h', 'rolling_mean_7d',
       'rolling_std_7d', 'rolling_max_24h', 'rolling_min_24h',
       'consumption_diff_1', 'consumption_diff_48', 'consumption_pct_change_1',
       'consumption_pct_change_48', 'season_Spring', 'season_Summer',
       'season_Winter', 'temp_sq', 'humidity_sq', 'hour_x_is_weekend',
       'hour_x_is_holiday', 'hour_x_dow', 'hour_x_month', 'is_weekend_x_month',
       'is_holiday_x_month', 'hour_x_temp', 'hour_x_humidity', 'hour_x_wind',
  

### Train test split  
We make sure to not shuffle the dataset and split it by the year

In [256]:
train_set = df[df['year'] < 2024].copy()

In [258]:
train_set

,Consommation,Zone_A,Zone_B,Zone_C,Vacances de la Toussaint,Vacances de Noël,Vacances d'Hiver,Vacances de Printemps,Vacances d'Été,public_holidays,...,is_weekend_x_season_Summer,is_holiday_x_season_Summer,hour_x_season_Winter,is_weekend_x_season_Winter,is_holiday_x_season_Winter,temp_x_humidity,temp_x_wind,humidity_x_wind,HDD,CDD
336,76986.0,0,0,0,0,0,0,0,0,0,...,0,0,0.0,0,0,-0.656254,-0.012465,4.407337,18.043083,0.0
337,75624.0,0,0,0,0,0,0,0,0,0,...,0,0,0.5,0,0,-1.417950,-0.052270,8.532836,18.093199,0.0
338,73317.0,0,0,0,0,0,0,0,0,0,...,0,0,1.0,0,0,-2.177823,-0.045113,4.783419,18.143315,0.0
339,74096.0,0,0,0,0,0,0,0,0,0,...,0,0,1.5,0,0,-1.666668,-0.061585,8.512510,18.109808,0.0
340,73854.0,0,0,0,0,0,0,0,0,0,...,0,0,2.0,0,0,-1.156723,-0.024581,4.883875,18.076301,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52555,53329.0,1,1,1,0,1,0,0,0,0,...,0,0,21.5,1,0,18.247310,0.801887,7.157649,16.570215,0.0
52556,52719.0,1,1,1,0,1,0,0,0,0,...,0,0,22.0,1,0,17.705367,1.263331,11.780869,16.622085,0.0
52557,53518.0,1,1,1,0,1,0,0,0,0,...,0,0,22.5,1,0,17.620965,0.759583,7.296909,16.645644,0.0
52558,55564.0,1,1,1,0,1,0,0,0,0,...,0,0,23.0,1,0,17.528969,1.191704,11.795066,16.669202,0.0


In [259]:
test_set = df[df['year'] == 2024].copy()

In [260]:
Y_train = train_set["Consommation"]

Y_test = test_set['Consommation']


X_train = train_set.drop(["Consommation", "year"], axis=1)
X_test  = test_set.drop(["Consommation", "year"], axis=1)

### Model training 1

In [261]:
rfg = RandomForestRegressor(1).fit(X_train, Y_train)

r2 = rfg.score(X_test, Y_test)

y_pred = rfg.predict(X_test)

mae = mean_absolute_error(Y_test, y_pred)
mse = mean_squared_error(Y_test, y_pred)

In [262]:
r2

0.9957489816029089

In [263]:
params = rfg.get_params()
params

{'bootstrap': True,
 'ccp_alpha': 0.0,
 'criterion': 'squared_error',
 'max_depth': None,
 'max_features': 1.0,
 'max_leaf_nodes': None,
 'max_samples': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'n_estimators': 1,
 'n_jobs': None,
 'oob_score': False,
 'random_state': None,
 'verbose': 0,
 'warm_start': False}

In [215]:
mlflow.set_tracking_uri("http://localhost:5000")

In [216]:
mlflow.set_experiment("Test experiment 1")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1782137361062, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1782137361062, lifecycle_stage='active', name='Test experiment 1', tags={}, trace_location=None, workspace='default'>

In [219]:
with mlflow.start_run():
    mlflow.log_params(params)

    mlflow.log_metric('r2 score', r2)
    mlflow.log_metric('mean absolute error', mae)
    mlflow.log_metric('mean squared error', mse)

    mlflow.set_tag('Training Info', 'Random forest Regressor for conso_v3_linear.parquet')

    signature = infer_signature(X_test, y_pred)

    model_info = mlflow.sklearn.log_model(
        sk_model = rfg,
        artifact_path = 'rfg_conso_v3_linear',
        signature = signature,
        input_example = X_test.head(3),
        registered_model_name='random forest regressor'
    )

2026/06/22 17:17:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'random forest regressor'.
2026/06/22 17:18:10 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: random forest regressor, version 1
Created version '1' of model 'random forest regressor'.


🏃 View run wistful-loon-906 at: http://localhost:5000/#/experiments/1/runs/38770fddc40543d2903c87b3d3a10343
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [228]:
model = mlflow.pyfunc.load_model(
    "models:/random forest regressor/1"
)

In [229]:
model.predict(X_test)

array([53956., 53956., 53986., ..., 63595., 62472., 64618.],
      shape=(17568,))

### Model training 2

In [248]:
gbr = GradientBoostingRegressor(n_estimators=15).fit(X_train, Y_train)

In [249]:
r2_gbr = gbr.score(X_test, Y_test)

y_pred_gbr = gbr.predict(X_test)

mae_gbr = mean_absolute_error(Y_test, y_pred_gbr)
mse_gbr = mean_squared_error(Y_test, y_pred_gbr)
params_gbr = gbr.get_params()

In [252]:
r2_gbr

0.9412099225526875

In [251]:
with mlflow.start_run():
    mlflow.log_params(params_gbr)

    mlflow.log_metric('r2 score', r2_gbr)
    mlflow.log_metric('mean absolute error', mae_gbr)
    mlflow.log_metric('mean squared error', mse_gbr)

    mlflow.set_tag('Training Info', 'Gradient boosting Regressor for conso_v3_linear.parquet')

    signature = infer_signature(X_test, y_pred_gbr)

    model_info = mlflow.sklearn.log_model(
        sk_model = rfg,
        name = 'model',
        signature = signature,
        input_example = X_test.head(3),
        registered_model_name='gbr_conso_linear'
    )

/home/marwane/energy_peak_prediction/venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
Registered model 'gbr_conso_linear' already exists. Creating a new version of this model...
2026/06/22 19:00:37 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish cr

🏃 View run agreeable-boar-672 at: http://localhost:5000/#/experiments/1/runs/5aab7be0525b47a4a449affcb4563a77
🧪 View experiment at: http://localhost:5000/#/experiments/1


## MLFLow with iris dataset

In [12]:
mlflow.set_tracking_uri('http://127.0.0.1:5000/')
mlflow.set_experiment("Classification Iris")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1782213471272, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1782213471272, lifecycle_stage='active', name='Classification Iris', tags={}, trace_location=None, workspace='default'>

In [26]:
iris = datasets.load_iris(as_frame=True)

In [27]:
X = iris.data
Y = iris.target

In [28]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2)

In [53]:
knn = KNN(2).fit(X_train, Y_train)

In [55]:
knn_accuracy = knn.score(X_test, Y_test)
knn_params = knn.get_params()

In [56]:
knn_accuracy

0.9333333333333333

In [57]:
with mlflow.start_run():
    mlflow.log_params(knn_params)

    mlflow.log_metric('accuracy', knn_accuracy)
    mlflow.sklearn.log_model(
        knn, 
        'KNN',
        skops_trusted_types=[
            'sklearn.metrics._dist_metrics.EuclideanDistance64',
            'sklearn.neighbors._kd_tree.KDTree',
        ]

    )

2026/06/23 13:44:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run chill-grouse-989 at: http://127.0.0.1:5000/#/experiments/1/runs/609bd47b01714054b8868dfb0537a9ee
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [62]:
for k in [1, 5, 7, 9]:
    with mlflow.start_run(run_name=f"knn_k{k}"):
        knn = KNN(n_neighbors=k)
        knn.fit(X_train, Y_train)
        accuracy = knn.score(X_test, Y_test)

        mlflow.log_param("n_neighbors", k)        # le param qu'on fait varier
        mlflow.log_metric("accuracy", accuracy)   # le résultat à comparer
        mlflow.sklearn.log_model(knn, "KNN", skops_trusted_types=[
            'sklearn.metrics._dist_metrics.EuclideanDistance64',
            'sklearn.neighbors._kd_tree.KDTree',
        ])

2026/06/23 13:49:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run knn_k1 at: http://127.0.0.1:5000/#/experiments/1/runs/88c52608c3ef4d19a1d8c9167c08e4f8
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/06/23 13:50:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run knn_k5 at: http://127.0.0.1:5000/#/experiments/1/runs/4561b33fc249492abb994c6d6461ed4d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/06/23 13:50:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run knn_k7 at: http://127.0.0.1:5000/#/experiments/1/runs/c2ecb7260f9b446c91a48ab9e93ec72f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/06/23 13:50:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run knn_k9 at: http://127.0.0.1:5000/#/experiments/1/runs/dceba40af0384b1a9a17899bce9d521c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [13]:
experiment = mlflow.get_experiment_by_name("Classification Iris")

In [17]:
runs = mlflow.search_runs(
    experiment_ids = [experiment.experiment_id],
    order_by = ['metrics.accuracy'],
    max_results=1
)

In [21]:
best_run = runs.iloc[0]
best_run_id = best_run["run_id"]

In [22]:
model_uri = f"runs:/{best_run_id}/KNN"
best_model = mlflow.sklearn.load_model(model_uri)

In [30]:
best_model.score(X_test, Y_test)

1.0